# 01 — Data exploration for MSD Task04 Hippocampus


## Setup

Assumption: this notebook lives in:

```text
medical-segmentation-uncertainty/
└── notebooks/
    └── 01_data_exploration.ipynb
```

So imports use `../src` and data uses `../data`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

PROJECT_ROOT, SRC_DIR

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import (
    MSDConfig,
    ensure_msd_downloaded,
    get_labelled_cases,
    load_dataset_json,
    load_nifti,
    case_basic_stats,
    split_cases,
)

try:
    from transforms import get_train_transforms, get_val_transforms
    HAS_TRANSFORMS = True
except Exception as e:
    HAS_TRANSFORMS = False
    print("Could not import transforms.py yet:", repr(e))

%matplotlib inline

In [ ]:
# Reproducibility
SEED = 21

# Internal labelled split.
# Official MSD test cases are unlabelled, so metrics must use held-out labelled data.
VAL_FRAC = 0.15
TEST_FRAC = 0.15

cfg = MSDConfig(
    root_dir=PROJECT_ROOT / "data",
    task="Task04_Hippocampus",
    seed=SEED,
)

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "data_exploration"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

cfg

## Dataset metadata

This confirms task name, modality, labels, number of labelled training cases, and number of unlabelled test cases.

In [ ]:
# Run downloads/extracts the dataset.
ensure_msd_downloaded(cfg, download=True)

In [ ]:
info = load_dataset_json(cfg)

print("Keys:", info.keys())
print("Name:", info.get("name"))
print("Description:", info.get("description"))
print("Modality:", info.get("modality"))
print("Labels:", info.get("labels"))
print("Number of labelled training cases:", len(info.get("training", [])))
print("Number of unlabelled test cases:", len(info.get("test", [])))

## Check that every file exists


In [ ]:
cases = get_labelled_cases(cfg)
len(cases), cases[0]

In [ ]:
missing = []

for case in cases:
    image_path = Path(case["image"])
    label_path = Path(case["label"])

    if not image_path.exists():
        missing.append(image_path)
    if not label_path.exists():
        missing.append(label_path)

print("Missing files:", len(missing))
missing[:10]

## Build a case-level dataframe

This is the main exploration table. 

In [ ]:
rows = []

for i, case in enumerate(cases, start=1):
    stats = case_basic_stats(case)

    row = {
        "case_id": stats["case_id"],
        "shape_x": stats["image_shape"][0],
        "shape_y": stats["image_shape"][1],
        "shape_z": stats["image_shape"][2],
        "spacing_x": stats["spacing"][0],
        "spacing_y": stats["spacing"][1],
        "spacing_z": stats["spacing"][2],
        "label_values": stats["label_values"],
        "foreground_voxels": stats["foreground_voxels"],
        "foreground_fraction": stats["foreground_fraction"],
        "image_p01": stats["image_p01"],
        "image_p50": stats["image_p50"],
        "image_p99": stats["image_p99"],
    }

    for label_value, count in stats["label_counts"].items():
        row[f"label_{label_value}_voxels"] = count

    rows.append(row)

    if i % 50 == 0 or i == len(cases):
        print(f"Processed {i}/{len(cases)} cases")

df = pd.DataFrame(rows)
df.head()

In [ ]:
summary_csv = METRICS_DIR / "data_exploration_summary.csv"
df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)

In [ ]:
df.describe(include="all")

## Check shapes and spacings

Do we need cropping, padding, resampling, and sliding-window inference?

In [ ]:
df[["shape_x", "shape_y", "shape_z"]].describe()

In [ ]:
df[["spacing_x", "spacing_y", "spacing_z"]].describe()

In [ ]:
ax = df[["shape_x", "shape_y", "shape_z"]].boxplot(figsize=(7, 5))
ax.set_title("Image shape distribution")
ax.set_ylabel("voxels")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "image_shape_distribution.png", dpi=200)
plt.show()

In [ ]:
ax = df[["spacing_x", "spacing_y", "spacing_z"]].boxplot(figsize=(7, 5))
ax.set_title("Voxel spacing distribution")
ax.set_ylabel("mm")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "voxel_spacing_distribution.png", dpi=200)
plt.show()

## Label values and class balance

For Task04 Hippocampus:

```text
0 = background
1 = anterior hippocampus
2 = posterior hippocampus
```

In [ ]:
all_label_values = sorted(set(v for values in df["label_values"] for v in values))
all_label_values

In [ ]:
label_cols = [c for c in df.columns if c.startswith("label_") and c.endswith("_voxels")]
df[label_cols].describe()

In [ ]:
ax = df["foreground_fraction"].hist(bins=30, figsize=(7, 5))
ax.set_title("Foreground fraction per case")
ax.set_xlabel("foreground voxels / total voxels")
ax.set_ylabel("number of cases")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "foreground_fraction_histogram.png", dpi=200)
plt.show()

## Per-class foreground volume


In [ ]:
total_voxels = df["shape_x"] * df["shape_y"] * df["shape_z"]

for class_id in [1, 2]:
    voxel_col = f"label_{class_id}_voxels"
    frac_col = f"label_{class_id}_fraction"

    if voxel_col in df.columns:
        df[frac_col] = df[voxel_col] / total_voxels
    else:
        df[frac_col] = 0.0

df[["label_1_fraction", "label_2_fraction"]].describe()

In [ ]:
ax = df[["label_1_fraction", "label_2_fraction"]].boxplot(figsize=(7, 5))
ax.set_title("Per-class foreground fraction")
ax.set_ylabel("fraction of total voxels")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_class_foreground_fraction.png", dpi=200)
plt.show()

## Visualize image, label, and overlay in all three planes


In [ ]:
def normalize_slice(x):
    lo, hi = np.percentile(x, [1, 99])
    return np.clip((x - lo) / (hi - lo + 1e-8), 0, 1)


def show_case(case, save_path=None):
    image, _ = load_nifti(case["image"])
    label, _ = load_nifti(case["label"])

    coords = np.argwhere(label > 0)
    if len(coords) == 0:
        raise ValueError(f"No foreground label found for case: {case['case_id']}")

    # Center of foreground bounding box / mass.
    center = coords.mean(axis=0).astype(int)
    x, y, z = center

    slices = [
        ("Sagittal", image[x, :, :], label[x, :, :]),
        ("Coronal", image[:, y, :], label[:, y, :]),
        ("Axial", image[:, :, z], label[:, :, z]),
    ]

    fig, axes = plt.subplots(3, 3, figsize=(10, 10))

    for row, (name, img_slice, lbl_slice) in enumerate(slices):
        axes[row, 0].imshow(np.rot90(normalize_slice(img_slice)), cmap="gray")
        axes[row, 0].set_title(f"{name} MRI")

        axes[row, 1].imshow(np.rot90(lbl_slice), interpolation="nearest")
        axes[row, 1].set_title(f"{name} label")

        axes[row, 2].imshow(np.rot90(normalize_slice(img_slice)), cmap="gray")
        masked = np.ma.masked_where(lbl_slice == 0, lbl_slice)
        axes[row, 2].imshow(np.rot90(masked), alpha=0.45, interpolation="nearest")
        axes[row, 2].set_title(f"{name} overlay")

        for col in range(3):
            axes[row, col].axis("off")

    fig.suptitle(case["case_id"])
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200)

    plt.show()

In [ ]:
for case in cases[:5]:
    print(case["case_id"])
    show_case(case, save_path=FIGURES_DIR / f"{case['case_id']}_three_plane_overlay.png")

In [ ]:
smallest_fg_id = df.sort_values("foreground_voxels").iloc[0]["case_id"]
largest_fg_id = df.sort_values("foreground_voxels").iloc[-1]["case_id"]

case_map = {case["case_id"]: case for case in cases}

print("Smallest foreground:", smallest_fg_id)
show_case(case_map[smallest_fg_id], save_path=FIGURES_DIR / f"{smallest_fg_id}_smallest_foreground_overlay.png")

print("Largest foreground:", largest_fg_id)
show_case(case_map[largest_fg_id], save_path=FIGURES_DIR / f"{largest_fg_id}_largest_foreground_overlay.png")

## Compute foreground bounding boxes


In [ ]:
def foreground_bbox(label):
    coords = np.argwhere(label > 0)
    if len(coords) == 0:
        return None

    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    size = maxs - mins + 1

    return mins, maxs, size

In [ ]:
bbox_rows = []

for case in cases:
    label, _ = load_nifti(case["label"])
    bbox = foreground_bbox(label)

    if bbox is None:
        bbox_rows.append({
            "case_id": case["case_id"],
            "bbox_x": 0,
            "bbox_y": 0,
            "bbox_z": 0,
        })
    else:
        _, _, size = bbox
        bbox_rows.append({
            "case_id": case["case_id"],
            "bbox_x": int(size[0]),
            "bbox_y": int(size[1]),
            "bbox_z": int(size[2]),
        })

bbox_df = pd.DataFrame(bbox_rows)
bbox_df.describe()

In [ ]:
bbox_csv = METRICS_DIR / "foreground_bounding_boxes.csv"
bbox_df.to_csv(bbox_csv, index=False)
print("Saved:", bbox_csv)

In [ ]:
ax = bbox_df[["bbox_x", "bbox_y", "bbox_z"]].boxplot(figsize=(7, 5))
ax.set_title("Foreground bounding box size")
ax.set_ylabel("voxels")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "foreground_bounding_box_size.png", dpi=200)
plt.show()

In [ ]:
bbox_df[["bbox_x", "bbox_y", "bbox_z"]].quantile([0.5, 0.75, 0.9, 0.95, 1.0])

## Check train/validation/internal-test split balance


In [ ]:
train_cases, val_cases, test_cases = split_cases(
    cases=cases,
    seed=SEED,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
)

print("Train:", len(train_cases))
print("Val:", len(val_cases))
print("Internal test:", len(test_cases))

In [ ]:
train_ids = {case["case_id"] for case in train_cases}
val_ids = {case["case_id"] for case in val_cases}
test_ids = {case["case_id"] for case in test_cases}

def assign_split(case_id):
    if case_id in train_ids:
        return "train"
    if case_id in val_ids:
        return "val"
    if case_id in test_ids:
        return "test"
    return "unknown"

df["split"] = df["case_id"].apply(assign_split)
df["split"].value_counts()

In [ ]:
df.groupby("split")[["foreground_fraction", "shape_x", "shape_y", "shape_z"]].describe()

In [ ]:
ax = df.boxplot(column="foreground_fraction", by="split", figsize=(7, 5))
ax.set_title("Foreground fraction by split")
plt.suptitle("")
ax.set_ylabel("foreground fraction")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "foreground_fraction_by_split.png", dpi=200)
plt.show()

## MRI intensity distribution


In [ ]:
df[["image_p01", "image_p50", "image_p99"]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["image_p50"], df["image_p99"])
ax.set_xlabel("median intensity")
ax.set_ylabel("99th percentile intensity")
ax.set_title("MRI intensity distribution per case")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mri_intensity_distribution.png", dpi=200)
plt.show()

## Preprocessing transforms

This checks whether the transform pipeline produces the shapes and label values we expect before training.

In [ ]:
if not HAS_TRANSFORMS:
    print("transforms.py is not available. Skip this section until src/transforms.py exists.")
else:
    val_tf = get_val_transforms(
        pixdim=(1.0, 1.0, 1.0),
        crop_margin=5,
    )

    sample = val_tf(cases[0])
    print("Validation image shape:", sample["image"].shape)
    print("Validation label shape:", sample["label"].shape)
    print("Validation label values:", np.unique(sample["label"].detach().cpu().numpy()))

In [ ]:
if not HAS_TRANSFORMS:
    print("transforms.py is not available. Skip this section until src/transforms.py exists.")
else:
    train_tf = get_train_transforms(
        spatial_size=(32, 32, 32),
        pixdim=(1.0, 1.0, 1.0),
        num_samples=4,
        pos=2.0,
        neg=1.0,
        crop_margin=5,
    )

    patches = train_tf(cases[0])
    print("Number of patches:", len(patches))

    patch = patches[0]
    print("Training patch image shape:", patch["image"].shape)
    print("Training patch label shape:", patch["label"].shape)
    print("Training patch label values:", np.unique(patch["label"].detach().cpu().numpy()))